# Microbial H₂ Consumption — Plotting More Experiments Together

Compares hydrogen consumption across a set of experiments run at different pressures (**60, 100, 200 and 400 bar**).

**Data.** Each experiment is stored as `.npy` arrays (loaded in §1):
- `time` — time in minutes
- `volume_1` — **Channel 1 / biotic** (microbial) curve
- `volume_2` — **Channel 2 / sterile** control curve

**Pipeline.**
1. Load the raw `.npy` arrays.
2. Trim each run to its minimum and zero it (`trim_and_normalize_set`).
3. Convert volume (ml) → amount of H₂ (mmol) using a pressure-specific density.
4. Visualise volumes, mmol curves, and net microbial consumption (biotic − sterile).
5. Quantify the *active* consumption window (time to reach 5 % and 90 % of total uptake).

In [ ]:
# All imports for the notebook live here.
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.signal import savgol_filter

from IPython.display import display


## 1 · Load raw experiment data

Reads every `.npy` file in the folder and binds each one to a global variable named after the file . Do not forget to change the file path.


In [ ]:
def load_npy_as_variables(folder_path):
    folder = Path(folder_path)

    for file_path in folder.glob('*.npy'):
        # Take the file name without the ".npy" extension
        base_name = file_path.stem

        # 1. Replace hyphens and spaces with underscores (_) so the name is valid in Python
        var_name = base_name.replace('-', '_').replace(' ', '_')

        # 2. If the name starts with a digit (e.g. 2026...), prepend a "v_"
        if var_name[0].isdigit():
            var_name = f"v_{var_name}"

        # 3. Create the variable directly in the global namespace!
        globals()[var_name] = np.load(file_path)

        print(f"Done! Created variable: {var_name}")

# --- HOW TO USE IT ---
load_npy_as_variables('/Users/adaciortan/Desktop/experiment data/daca ai fii un stick/the column')

# AFTER RUNNING THE FUNCTION, YOU CAN ACCESS THEM DIRECTLY:
# print(v_2026_02_25_h2_400bar_SRB_volume_1)


## 2 · Trim & normalise each run

In [ ]:
def trim_and_normalize_set(time_arr, vol1_arr, vol2_arr):
    """
    Synchronise and normalise a set of 3 arrays based on the minimum of Volume 1.
    """
    # 1. Find the index of the minimum value in Volume 1
    min_idx = np.argmin(vol1_arr)

    # 2. Trim all 3 arrays from that index to the end
    t_trim = time_arr[min_idx:]
    v1_trim = vol1_arr[min_idx:]
    v2_trim = vol2_arr[min_idx:]

    # 3. Subtract the minimum from each trimmed array so it starts at 0
    # We use np.min() to make sure the lowest point becomes exactly 0.0
    t_norm = t_trim - np.min(t_trim)
    v1_norm = v1_trim - np.min(v1_trim)
    v2_norm = v2_trim - np.min(v2_trim)

    return t_norm, v1_norm, v2_norm


In [ ]:
# 1. Experiment 25 Feb 2026 (400 bar)
time_26_02_25_clean, vol1_26_02_25_clean, vol2_26_02_25_clean = trim_and_normalize_set(
    v_2026_02_25_h2_400bar_SRB_time, 
    v_2026_02_25_h2_400bar_SRB_volume_1, 
    v_2026_02_25_h2_400bar_SRB_volume_2
)

# 2. Experiment 01 Jul 2025 (100 bar)
time_25_07_01_clean, vol1_25_07_01_clean, vol2_25_07_01_clean = trim_and_normalize_set(
    v_2025_07_01_h2_100bar_SRB_time, 
    v_2025_07_01_h2_100bar_SRB_volume_1, 
    v_2025_07_01_h2_100bar_SRB_volume_2
)

# 3. Experiment 10 Feb 2026 (200 bar)
time_26_02_10_clean, vol1_26_02_10_clean, vol2_26_02_10_clean = trim_and_normalize_set(
    v_2026_02_10_h2_200bar_SRB_time, 
    v_2026_02_10_h2_200bar_SRB_volume_1, 
    v_2026_02_10_h2_200bar_SRB_volume_2
)

# 4. Experiment 23 Oct 2025 (60 bar)
time_25_10_23_clean, vol1_25_10_23_clean, vol2_25_10_23_clean = trim_and_normalize_set(
    v_2025_10_23_h2_60bar_SRB_time, 
    v_2025_10_23_h2_60bar_SRB_volume_1, 
    v_2025_10_23_h2_60bar_SRB_volume_2
)

# 5. Experiment 18 Jun 2025 (100 bar)
time_25_06_18_clean, vol1_25_06_18_clean, vol2_25_06_18_clean = trim_and_normalize_set(
    v_2025_06_18_h2_100bar_SRB_time, 
    v_2025_06_18_h2_100bar_SRB_volume_1, 
    v_2025_06_18_h2_100bar_SRB_volume_2
)

# 6. Experiment 29 Jan 2026 (400 bar)
# Convert time from days to minutes
v_2026_01_29_h2_400bar_SRB_time_min = v_2026_01_29_h2_400bar_SRB_time * 24 * 60
time_26_01_29_clean, vol1_26_01_29_clean, vol2_26_01_29_clean = trim_and_normalize_set(
    v_2026_01_29_h2_400bar_SRB_time_min, 
    v_2026_01_29_h2_400bar_SRB_volume_1, 
    v_2026_01_29_h2_400bar_SRB_volume_2
)

# 7. Experiment 11 Nov 2025 (200 bar)
v_2025_11_11_h2_200bar_SRB_time_gen = np.arange(0, 18324)
time_25_11_11_clean, vol1_25_11_11_clean, vol2_25_11_11_clean = trim_and_normalize_set(
    v_2025_11_11_h2_200bar_SRB_time_gen, 
    v_2025_11_11_h2_200bar_SRB_volume_1, 
    v_2025_11_11_h2_200bar_SRB_volume_2
)

# 8. Experiment 12 Mar 2026 (200 bar)
time_26_03_12_clean, vol1_26_03_12_clean, vol2_26_03_12_clean = trim_and_normalize_set(
    v_2026_03_12_h2_200bar_SRB_time, 
    v_2026_03_12_h2_200bar_SRB_volume_1, 
    v_2026_03_12_h2_200bar_SRB_volume_2
)

# 9. Experiment 02 Apr 2026 (200 bar) - Mentioned in your variable log
time_26_04_02_clean, vol1_26_04_02_clean, vol2_26_04_02_clean = trim_and_normalize_set(
    v_2026_04_02_h2_200bar_SRB_time, 
    v_2026_04_02_h2_200bar_SRB_volume_1, 
    v_2026_04_02_h2_200bar_SRB_volume_2
)


## 3 · Quick look — normalised volumes

In [ ]:
def plot_all_experiments():
    plt.figure(figsize=(14, 8))

    # List of experiments based on the variables created earlier
    experiments = [
        ("25 Feb 2026 (400 bar)", time_26_02_25_clean, vol1_26_02_25_clean, vol2_26_02_25_clean),
        ("01 Jul 2025 (100 bar)", time_25_07_01_clean, vol1_25_07_01_clean, vol2_25_07_01_clean),
        ("10 Feb 2026 (200 bar)", time_26_02_10_clean, vol1_26_02_10_clean, vol2_26_02_10_clean),
        ("23 Oct 2025 (60 bar)", time_25_10_23_clean, vol1_25_10_23_clean, vol2_25_10_23_clean),
        ("18 Jun 2025 (100 bar)", time_25_06_18_clean, vol1_25_06_18_clean, vol2_25_06_18_clean),
        ("29 Jan 2026 (400 bar)", time_26_01_29_clean, vol1_26_01_29_clean, vol2_26_01_29_clean),
        ("11 Nov 2025 (200 bar)", time_25_11_11_clean, vol1_25_11_11_clean, vol2_25_11_11_clean),
        ("12 Mar 2026 (200 bar)", time_26_03_12_clean, vol1_26_03_12_clean, vol2_26_03_12_clean),
        ("02 Apr 2026 (200 bar)", time_26_04_02_clean, vol1_26_04_02_clean, vol2_26_04_02_clean)
    ]

    for label, t, v1, v2 in experiments:
        line = plt.plot(t, v1, label=f"{label} (V1)", linewidth=1.5)
        color = line[0].get_color()
        plt.plot(t, v2, label=f"{label} (V2)", linestyle='--', color=color, alpha=0.7, linewidth=1.2)

    plt.title('Experimental Volume Comparison (Trimmed & Normalized)', fontsize=14)
    plt.xlabel('Time (minutes)', fontsize=12)
    plt.ylabel('Normalized Volume', fontsize=12)

    # Legend placed outside so it does not cover the data
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small', ncol=1)
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.tight_layout()

    plt.show()

# Call the function to display the plot
plot_all_experiments()


## 4 · Convert volume (ml) → amount of H₂ (mmol)

Uses a pressure-specific H₂ density and the molar mass to turn measured volumes into mmol.

In [ ]:
def convert_ml_to_mmol(volume_array_ml, pressure_bar):
    """
    Convert an array of volumes from ml to mmol of H2,
    accounting for the pressure-specific density.
    """
    molar_mass_h2 = 2.016  # Molar mass of H2 in g/mol (i.e. mg/mmol)

    if pressure_bar == 100:
        density = 7.392   # kg/m^3 (equivalent to g/L)
    elif pressure_bar == 200:
        density = 14.030
    elif pressure_bar == 400:
        density = 25.743
    elif pressure_bar == 60:
        density = 4.534
    else:
        raise ValueError("Unknown pressure. Please choose 60, 100, 200 or 400.")

    # Calculation: (Volume * Density) / Molar Mass
    return volume_array_ml * (density / molar_mass_h2)

# --- Conversion to mmol for all experiments ---

# 400 bar
mmol1_26_02_25 = convert_ml_to_mmol(vol1_26_02_25_clean, 400)
mmol2_26_02_25 = convert_ml_to_mmol(vol2_26_02_25_clean, 400)

mmol1_26_01_29 = convert_ml_to_mmol(vol1_26_01_29_clean, 400)
mmol2_26_01_29 = convert_ml_to_mmol(vol2_26_01_29_clean, 400)

# 200 bar
mmol1_26_02_10 = convert_ml_to_mmol(vol1_26_02_10_clean, 200)
mmol2_26_02_10 = convert_ml_to_mmol(vol2_26_02_10_clean, 200)

mmol1_25_11_11 = convert_ml_to_mmol(vol1_25_11_11_clean, 200)
mmol2_25_11_11 = convert_ml_to_mmol(vol2_25_11_11_clean, 200)

mmol1_26_03_12 = convert_ml_to_mmol(vol1_26_03_12_clean, 200)
mmol2_26_03_12 = convert_ml_to_mmol(vol2_26_03_12_clean, 200)

mmol1_26_04_02 = convert_ml_to_mmol(vol1_26_04_02_clean, 200)
mmol2_26_04_02 = convert_ml_to_mmol(vol2_26_04_02_clean, 200)

# 100 bar
mmol1_25_07_01 = convert_ml_to_mmol(vol1_25_07_01_clean, 100)
mmol2_25_07_01 = convert_ml_to_mmol(vol2_25_07_01_clean, 100)

mmol1_25_06_18 = convert_ml_to_mmol(vol1_25_06_18_clean, 100)
mmol2_25_06_18 = convert_ml_to_mmol(vol2_25_06_18_clean, 100)

# 60 bar
mmol1_25_10_23 = convert_ml_to_mmol(vol1_25_10_23_clean, 60)
mmol2_25_10_23 = convert_ml_to_mmol(vol2_25_10_23_clean, 60)


## 5 · Visualise H₂ amounts

*All runs, biotic (solid) vs. sterile (dashed), in mmol.*

In [ ]:
# --- Visualise the data in mmol ---

def plot_mmol_comparison():
    plt.figure(figsize=(14, 8))

    data_to_plot = [
        ("25 Feb 26 (400b)", time_26_02_25_clean, mmol1_26_02_25, mmol2_26_02_25),
        ("29 Jan 26 (400b)", time_26_01_29_clean, mmol1_26_01_29, mmol2_26_01_29),
        ("10 Feb 26 (200b)", time_26_02_10_clean, mmol1_26_02_10, mmol2_26_02_10),
        ("11 Nov 25 (200b)", time_25_11_11_clean, mmol1_25_11_11, mmol2_25_11_11),
        ("12 Mar 26 (200b)", time_26_03_12_clean, mmol1_26_03_12, mmol2_26_03_12),
        ("02 Apr 26 (200b)", time_26_04_02_clean, mmol1_26_04_02, mmol2_26_04_02),
        ("01 Jul 25 (100b)", time_25_07_01_clean, mmol1_25_07_01, mmol2_25_07_01),
        ("18 Jun 25 (100b)", time_25_06_18_clean, mmol1_25_06_18, mmol2_25_06_18),
        ("23 Oct 25 (60b)",  time_25_10_23_clean, mmol1_25_10_23, mmol2_25_10_23)
    ]

    for label, t, m1, m2 in data_to_plot:
        p = plt.plot(t, m1, label=f"{label} S1", linewidth=1.5)
        plt.plot(t, m2, label=f"{label} S2", linestyle='--', color=p[0].get_color(), alpha=0.6)

    plt.title('$H_2$ Production - Comparison in amount of substance (mmol)', fontsize=14)
    plt.xlabel('Time (minutes)', fontsize=12)
    plt.ylabel('Amount of $H_2$ (mmol)', fontsize=12)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    plt.grid(True, linestyle=':', alpha=0.7)
    plt.tight_layout()
    plt.show()

plot_mmol_comparison()


*Biotic vs. sterile channels, plus net microbial consumption (biotic − sterile).*

In [ ]:
# Organise the data for an efficient loop
all_data = [
    ("60 bar - run 1",  time_25_10_23_clean, mmol1_25_10_23, mmol2_25_10_23),
    ("100 bar - run 1", time_25_06_18_clean, mmol1_25_06_18, mmol2_25_06_18),
    ("100 bar - run 2", time_25_07_01_clean, mmol1_25_07_01, mmol2_25_07_01),
    ("200 bar - run 1", time_26_02_10_clean, mmol1_26_02_10, mmol2_26_02_10),
    ("200 bar - run 2", time_26_04_02_clean, mmol1_26_04_02, mmol2_26_04_02),
    ("400 bar - run 1", time_26_01_29_clean, mmol1_26_01_29, mmol2_26_01_29),
    ("400 bar - run 2", time_26_02_25_clean, mmol1_26_02_25, mmol2_26_02_25)
]

# Convert time from minutes to days (1 day = 1440 min)
all_data = [(label, t / 1440.0, m1, m2) for label, t, m1, m2 in all_data]

# Global style settings
plt.rcParams.update({'font.size': 10})

# --- FIGURE 1: Sterile vs Microbial (individual) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))

for label, t, m1, m2 in all_data:

    ax1.plot(t, m1, label=label, linewidth=1.5)
    ax2.plot(t, m2, label=label, linewidth=1.5)

ax2.set_title("Sterile Curve (Channel 2)")
ax2.set_ylabel("Amount of Hydrogen [mmol]")
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper left', bbox_to_anchor=(1, 1))

ax1.set_title("Biotic Curve (Channel 1)")
ax1.set_ylabel("Amount of Hydrogen [mmol]")
ax1.set_xlabel("Time [days]")
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# --- FIGURE 2: Net Consumption ---
plt.figure(figsize=(10, 6))

for label, t, m1, m2 in all_data:
    # Plot the net difference for every run
    plt.plot(t, m1 - m2, label=f"Net: {label}", linewidth=1.8)

plt.axhline(0, color='black', linestyle='-', alpha=0.3)  # Zero reference line
plt.title("Microbial Hydrogen Consumption (Channel 1 - Channel 2)")
plt.xlabel("Time [days]")
plt.ylabel("Consumed H2 [mmol]")
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()


*Per-pressure plots, written to the `H2_plots/` folder on disk.*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import os

# ─────────────────────────────────────────────────────────────────
# YOUR DATA VARIABLES (plug in your actual arrays here)
# ─────────────────────────────────────────────────────────────────
# Replace these with your actual time/mmol arrays:
# time_26_02_25_clean, mmol1_26_02_25, mmol2_26_02_25, etc.

# ─────────────────────────────────────────────────────────────────
# DATA REGISTRY — group experiments by pressure
# ─────────────────────────────────────────────────────────────────
pressure_groups = {
    "400 bar": [
        ("25 Feb 2026", time_26_02_25_clean, mmol1_26_02_25, mmol2_26_02_25),
        ("29 Jan 2026", time_26_01_29_clean, mmol1_26_01_29, mmol2_26_01_29),
    ],
    "200 bar": [
        ("10 Feb 2026", time_26_02_10_clean, mmol1_26_02_10, mmol2_26_02_10),
        ("02 Apr 2026", time_26_04_02_clean, mmol1_26_04_02, mmol2_26_04_02),
    ],
    "100 bar": [
        ("01 Jul 2025", time_25_07_01_clean, mmol1_25_07_01, mmol2_25_07_01),
        ("18 Jun 2025", time_25_06_18_clean, mmol1_25_06_18, mmol2_25_06_18),
    ],
    "60 bar": [
        ("23 Oct 2025", time_25_10_23_clean, mmol1_25_10_23, mmol2_25_10_23),
    ],
}

# ─────────────────────────────────────────────────────────────────
# STYLE SETTINGS
# ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "figure.dpi": 150,
})

PLOT_TYPES = [
    ("Microbial",   "mmol1",       "Microbial curves ({pressure})",   "Amount H₂ [mmol]"),
    ("Sterile",     "mmol2",       "Sterile curves ({pressure})",     "Amount H₂ [mmol]"),
    ("Difference",  "mmol1-mmol2", "Difference curves ({pressure})",  "Consumed H₂ [mmol]"),
]

OUTPUT_ROOT = "H2_plots"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# HELPER: pick a color palette for N curves
# ─────────────────────────────────────────────────────────────────
def get_colors(n, cmap_name="tab10"):
    cmap = cm.get_cmap(cmap_name)
    return [cmap(i / max(n - 1, 1)) for i in range(n)]


# ─────────────────────────────────────────────────────────────────
# HELPER: save one figure
# ─────────────────────────────────────────────────────────────────
def save_fig(fig, path):
    fig.savefig(path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"  Saved → {path}")


# ─────────────────────────────────────────────────────────────────
# 1.  PER-PRESSURE PLOTS
# ─────────────────────────────────────────────────────────────────
for pressure, experiments in pressure_groups.items():
    # Folder name: "400_bar", "200_bar", etc.
    folder_name = pressure.replace(" ", "_")
    folder_path = os.path.join(OUTPUT_ROOT, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    print(f"\n── {pressure} ──")
    colors = get_colors(len(experiments))

    for plot_type, data_key, title_template, ylabel in PLOT_TYPES:
        title = title_template.format(pressure=pressure)
        fig, ax = plt.subplots(figsize=(9, 5))

        for (exp_label, t, m1, m2), color in zip(experiments, colors):
            if data_key == "mmol1":
                y = m1
            elif data_key == "mmol2":
                y = m2
            else:  # mmol1-mmol2
                y = m1 - m2

            ax.plot(t, y, label=exp_label, color=color, linewidth=1.8)

        if data_key == "mmol1-mmol2":
            ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.4)

        ax.set_title(title)
        ax.set_xlabel("Time [minutes]")
        ax.set_ylabel(ylabel)
        ax.grid(True, linestyle="--", alpha=0.5)
        ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)

        filename = f"{plot_type}_{folder_name}.png"
        save_fig(fig, os.path.join(folder_path, filename))


# ─────────────────────────────────────────────────────────────────
# 2.  COMBINED PLOTS — all pressures together
# ─────────────────────────────────────────────────────────────────
combined_folder = os.path.join(OUTPUT_ROOT, "All_pressures_combined")
os.makedirs(combined_folder, exist_ok=True)
print("\n── All pressures combined ──")

# Flatten: (label_with_pressure, t, m1, m2)
all_experiments = [
    (f"{exp_label} ({pressure})", t, m1, m2)
    for pressure, experiments in pressure_groups.items()
    for (exp_label, t, m1, m2) in experiments
]
colors_all = get_colors(len(all_experiments), cmap_name="tab20")

for plot_type, data_key, title_template, ylabel in PLOT_TYPES:
    title = title_template.format(pressure="All Pressures")
    fig, ax = plt.subplots(figsize=(11, 6))

    for (exp_label, t, m1, m2), color in zip(all_experiments, colors_all):
        if data_key == "mmol1":
            y = m1
        elif data_key == "mmol2":
            y = m2
        else:
            y = m1 - m2

        ax.plot(t, y, label=exp_label, color=color, linewidth=1.6)

    if data_key == "mmol1-mmol2":
        ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.4)

    ax.set_title(title)
    ax.set_xlabel("Time [minutes]")
    ax.set_ylabel(ylabel)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0,
              fontsize=8)

    filename = f"{plot_type}_All_pressures.png"
    save_fig(fig, os.path.join(combined_folder, filename))

print("\n✓ All plots saved to:", OUTPUT_ROOT)


## 6 · Active consumption window (5 %–90 %)

Time from start until net consumption (biotic − sterile) reaches 5 % and 90 % of its total rise, per experiment — table plus one panel per run. *(This consolidates three earlier near-identical drafts of the same analysis.)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def time_to_fraction(t_days, consumed, frac, t_ignore_before=0.0):
    """Time (days) until consumed reaches `frac` of the total rise (baseline -> peak),
    counting only upward progress. Points before `t_ignore_before` (days) are excluded
    from the search for the baseline and the rise, but stay in the data otherwise."""
    t = np.asarray(t_days, float)
    y = np.asarray(consumed, float)
    use = t >= t_ignore_before
    t = t[use]; y = y[use]
    if len(y) < 2:
        return np.nan
    i_peak = int(np.argmax(y))
    total = y[i_peak] - y[0]
    if total <= 0:
        return np.nan
    y_pos = np.maximum.accumulate(y[:i_peak + 1])   # positive-slope-only envelope
    target = y[0] + frac * total
    idx = int(np.argmax(y_pos >= target))
    if idx == 0:
        return 0.0
    y0, y1, t0, t1 = y_pos[idx-1], y_pos[idx], t[idx-1], t[idx]
    tc = t0 if y1 == y0 else t0 + (target - y0) * (t1 - t0) / (y1 - y0)
    return float(tc - t[0])


def fraction_crossing(t_days, consumed, frac, t_ignore_before=0.0):
    t = np.asarray(t_days, float); y = np.asarray(consumed, float)
    use = t >= t_ignore_before
    t = t[use]; y = y[use]
    if len(y) < 2:
        return np.nan, np.nan
    i_peak = int(np.argmax(y)); total = y[i_peak] - y[0]
    if total <= 0:
        return np.nan, np.nan
    y_pos = np.maximum.accumulate(y[:i_peak + 1])    # positive-slope-only envelope
    target = y[0] + frac * total
    idx = int(np.argmax(y_pos >= target))
    if idx == 0:
        return float(t[0]), float(y[0])
    y0, y1, t0, t1 = y_pos[idx-1], y_pos[idx], t[idx-1], t[idx]
    tc = t0 if y1 == y0 else t0 + (target - y0) * (t1 - t0) / (y1 - y0)
    return float(tc), float(target)


# (label, pressure, time[min], biotic mmol, sterile mmol, ignore_before_days)
# Only the 25 Feb 2026 (400 bar) run ignores its first day in the start search.
experiments = [
    ("25 Feb 2026", 400, time_26_02_25_clean, mmol1_26_02_25, mmol2_26_02_25, 1.0),
    ("29 Jan 2026", 400, time_26_01_29_clean, mmol1_26_01_29, mmol2_26_01_29, 0.0),
    ("10 Feb 2026", 200, time_26_02_10_clean, mmol1_26_02_10, mmol2_26_02_10, 0.0),
    ("02 Apr 2026", 200, time_26_04_02_clean, mmol1_26_04_02, mmol2_26_04_02, 0.0),
    ("01 Jul 2025", 100, time_25_07_01_clean, mmol1_25_07_01, mmol2_25_07_01, 0.0),
    ("18 Jun 2025", 100, time_25_06_18_clean, mmol1_25_06_18, mmol2_25_06_18, 0.0),
    ("23 Oct 2025",  60, time_25_10_23_clean, mmol1_25_10_23, mmol2_25_10_23, 0.0),
]

# one distinct colour per experiment
colors = plt.cm.tab10(np.linspace(0, 1, len(experiments)))

rows = []
for label, P, t_min, m1, m2, ignore in experiments:
    consumed = np.asarray(m1, float) - np.asarray(m2, float)   # biotic - sterile
    consumed = consumed - consumed[0]                          # start at zero
    t_days = np.asarray(t_min, float) / 1440.0                 # minutes -> days
    rows.append({
        "experiment": label,
        "pressure (bar)": P,
        "t to 5% (days)":  time_to_fraction(t_days, consumed, 0.05, ignore),
        "t to 90% (days)": time_to_fraction(t_days, consumed, 0.90, ignore),
    })

times_df = pd.DataFrame(rows)
display(times_df)

ncols = 3
nrows = int(np.ceil(len(experiments) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for ax, (label, P, t_min, m1, m2, ignore), color in zip(axes, experiments, colors):
    consumed = np.asarray(m1, float) - np.asarray(m2, float)
    consumed = consumed - consumed[0]
    t_days = np.asarray(t_min, float) / 1440.0
    t5,  y5  = fraction_crossing(t_days, consumed, 0.05, ignore)
    t90, y90 = fraction_crossing(t_days, consumed, 0.90, ignore)

    # full curve stays visible, including any ignored early data
    ax.plot(t_days, consumed, color=color, lw=1.5)


    if np.isfinite(t5) and np.isfinite(t90):
        ax.axvspan(t5, t90, color="green", alpha=0.15)
        ax.scatter([t5, t90], [y5, y90], color="black", zorder=5, s=25)
        ax.set_title(f"{label}  ({P} bar)\nactive {t5:.2f}-{t90:.2f} d  (Δ={t90-t5:.2f} d)", fontsize=10)
    else:
        ax.set_title(f"{label} ({P} bar) - no net consumption", fontsize=10)
    ax.set_xlabel("Time (days)"); ax.set_ylabel("Consumed H2 (mmol)")
    ax.grid(True, ls=":", alpha=0.5)

for ax in axes[len(experiments):]:   # hide unused panels
    ax.axis("off")
fig.suptitle("Active area (5%-90% of total consumption) per experiment", fontsize=14, y=1.005)
fig.tight_layout()
plt.show()
